# nl2cuda-kernel-agent — Colab 一站式验证 notebook

配合 `USAGE.md` **第一部分·路径 A** 使用。运行时选 **T4 GPU**（`代码执行程序 → 更改运行时类型 → T4 GPU`）。

**只有两个 cell，从上到下跑**：
1. **冒烟 cell**：clone 仓库 + 装 ninja + 跑内置 rbf（verify/bench），确认环境就绪。
2. **你的 case cell**：把本地 agent 产的 case 打包成 base64（见 USAGE A-2），填进 `CASE`/`B64` 两个空，一个 cell 跑完解包 + verify + bench。

> ⚠️ Colab 闲置约 90 分钟/断网会**重置运行时**（仓库、case、编译产物全丢）。所以把流程压进**一个 cell 一次性跑完**——报错重跑该 cell 即可（它幂等：仓库在就跳过 clone）。

## Cell 1 · 冒烟：确认环境（跑内置 rbf 样例）
首次 nvcc 编译 rbf 要等几分钟、中途无输出正常。rbf 的 verify 全 PASS 即环境就绪。

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('/content/nl2cuda-kernel-agent'):
    !git clone https://github.com/SilenceWanna/nl2cuda-kernel-agent.git
os.chdir('/content/nl2cuda-kernel-agent')
!pip install ninja -q
!python scripts/probe_env.py                      # 确认 GPU / CUDA / PyTorch
!python framework/smoke_test.py                   # 确认 nvcc + ninja 编译链路
!python skill/scripts/verify_case.py --case rbf   # 冒烟：前反向 5 种子应全 PASS
!python skill/scripts/bench_case.py  --case rbf   # 冒烟：加速比（rbf 参考前~1.10x/反~1.17x）

## Cell 2 · 你的 case：解包 + verify + bench（一次性）

先在**本地**让 agent 产出 `cases/<你的算法名>/`，按 USAGE **A-2** 打包成一行 base64。然后：
- 把 `CASE` 改成你的 case 名（如 `rmsnorm`，**不要留尖括号**）；
- 把 `B64` 粘上那一整行 base64（结尾通常是 `==`）；
- 跑本 cell。改一版就重打包、重跑本 cell（幂等，仓库在就跳过 clone）。

> 若 `bench` 报「短核假象警告」（baseline <1ms），在下面补一个 cell 放大规模复测：
> `!<规模ENV>=262144 python skill/scripts/bench_case.py --case 你的case名`（`<规模ENV>` 见该 case 的 `config.py`，如 RMSNorm 是 `RMS_B`）。

In [ ]:
import os, base64, tarfile, io
os.chdir('/content')
if not os.path.isdir('/content/nl2cuda-kernel-agent'):
    !git clone https://github.com/SilenceWanna/nl2cuda-kernel-agent.git
os.chdir('/content/nl2cuda-kernel-agent')
!pip install ninja -q

CASE = '你的算法名'                        # ← 改成实际 case 名，如 rmsnorm（不要留尖括号）
B64  = '在此粘贴 USAGE A-2 那一整行 base64'  # ← 一整行，结尾通常是 ==

with tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64))) as t:
    t.extractall('.')
print('case 文件:', os.listdir(f'cases/{CASE}'))
!python skill/scripts/verify_case.py --case {CASE}
!python skill/scripts/bench_case.py  --case {CASE}